In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0025611.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0031005.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0032987.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0027620.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0033874.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0024688.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset/mel/test/ISIC_0027163.jpg
/kaggle/input/datasets/maimoonakhan/balanced-and-augmen

In [2]:
!pip install timm

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import gc

# 1. EXTREME PARANOIA CLEANUP (Wipes previous hidden memory)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 2. LOCK TO ONE GPU (Zero memory leaks)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"System completely locked to: {device}")

# 3. RAM-SAFE DATA PIPELINE (224x224 images)
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_dir = '/kaggle/input/datasets/maimoonakhan/balanced-and-augmented-version-of-the-ham10000/Balanced Ham10000 Dataset'
dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)

# batch_size 8 is ultra-safe for 15GB VRAM. num_workers=0 is MANDATORY.
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
print(f"Loaded {len(dataset)} images. Ready.")

# 4. THE MODEL
model = timm.create_model('convnextv2_base.fcmae_ft_in22k_in1k', pretrained=True, num_classes=7)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 5. BULLETPROOF TRAINING LOOP (With AMP)
scaler = torch.amp.GradScaler('cuda') 
epochs = 5

for epoch in range(epochs):
    model.train()
    correct = 0
    total = 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # 16-bit Math to save VRAM
        with torch.amp.autocast(device_type='cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Math for accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # MANUAL TRASH REMOVAL (Guarantees RAM stays flat)
        del images, labels, outputs
        
        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Step [{i+1}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    epoch_acc = 100 * correct / total
    print(f"\n--- EPOCH {epoch+1} FINISHED | Accuracy: {epoch_acc:.2f}% ---\n")

# 6. FINAL SAVE
save_path = '/kaggle/working/convnextv2_skin_cancer_bulletproof.pth'
torch.save(model.state_dict(), save_path)
print(f"Success! Model securely saved to: {save_path}")

System completely locked to: cuda:0
Loaded 4766 images. Ready.


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

Epoch [1/5] | Step [50/596] | Loss: 2.2330
Epoch [1/5] | Step [100/596] | Loss: 1.2002
Epoch [1/5] | Step [150/596] | Loss: 0.8496
Epoch [1/5] | Step [200/596] | Loss: 1.9606
Epoch [1/5] | Step [250/596] | Loss: 0.9588
Epoch [1/5] | Step [300/596] | Loss: 1.0463
Epoch [1/5] | Step [350/596] | Loss: 0.3285
Epoch [1/5] | Step [400/596] | Loss: 0.2243
Epoch [1/5] | Step [450/596] | Loss: 0.7356
Epoch [1/5] | Step [500/596] | Loss: 0.4352
Epoch [1/5] | Step [550/596] | Loss: 0.4016

--- EPOCH 1 FINISHED | Accuracy: 70.96% ---

Epoch [2/5] | Step [50/596] | Loss: 0.2561
Epoch [2/5] | Step [100/596] | Loss: 0.0950
Epoch [2/5] | Step [150/596] | Loss: 0.3786
Epoch [2/5] | Step [200/596] | Loss: 0.0158
Epoch [2/5] | Step [250/596] | Loss: 0.2602
Epoch [2/5] | Step [300/596] | Loss: 0.3569
Epoch [2/5] | Step [350/596] | Loss: 0.3330
Epoch [2/5] | Step [400/596] | Loss: 0.0371
Epoch [2/5] | Step [450/596] | Loss: 0.8188
Epoch [2/5] | Step [500/596] | Loss: 0.1205
Epoch [2/5] | Step [550/596] | L